In [ ]:
from pathlib import Path
import os
import ast
import pandas as pd
import random
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import math 
from pathlib import Path
import torch
import transformers
import accelerate
import triton
import kernels
import PIL
from transformers import GptOssConfig
import warnings 
warnings.filterwarnings("ignore")
from transformers import AutoTokenizer, AutoModelForCausalLM
import glob
import json

In [ ]:
!pip install -U transformers accelerate kernels "triton>=3.4.0"

In [ ]:
!nvidia-smi

In [ ]:
model_id = "openai/gpt-oss-20b"
config = GptOssConfig.from_pretrained(model_id)
print("Model configuration loaded.")
print("Quantization configuration:")
print(config.quantization_config)

In [ ]:
model_id = "openai/gpt-oss-20b"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)
print(model.hf_device_map)

In [ ]:
print(os.listdir("/kaggle/input/datasets/aryansingh1964/project-3-nlp/dataset"))

In [ ]:


INPUT_DIR = Path("/kaggle/input")

print("Input folder exists:", INPUT_DIR.exists())
print("Contents of /kaggle/input:")

for item in INPUT_DIR.iterdir():
    print(item.name, "->", "folder" if item.is_dir() else "file")


In [ ]:
INPUT_DIR = Path("/kaggle/input")

zip_files = list(INPUT_DIR.rglob("*.zip"))

print("ZIP files found:", len(zip_files))

for file in zip_files:
    print(file)


In [ ]:

from pathlib import Path

INPUT_DIR = Path("/kaggle/input/datasets/aryansingh1964")

zip_files = list(INPUT_DIR.glob("*.zip"))

for file in zip_files:
    print("ZIP file:", file)


In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/input/datasets/aryansingh1964/project-3-nlp/dataset")

data_dir = ROOT / "data"
img_dir = ROOT / "img_text"

print("Root exists:", ROOT.exists())
print("Data folder exists:", data_dir.exists())
print("Image-text folder exists:", img_dir.exists())

In [ ]:
ROOT = Path("/kaggle/input/datasets/aryansingh1964/project-3-nlp/dataset")
DATA_DIR = ROOT/ "data"
IMG_DIR = ROOT/ "img_text"
# Checking the main folder are present or not
print("Dataset root exists: ",ROOT.exists())
print("Annotation folder exists: ",DATA_DIR.exists())
print("Image-Text folder exists: ",IMG_DIR.exists())

In [ ]:
# Now inspecting  annotatio folders and split files
for folder in sorted(DATA_DIR.iterdir()):
    if folder.is_dir():
        print(f"\nfolder: {folder.name}")
        for split in ["train.txt","val.txt","test.txt"]:
            file_path = folder/split
            if file_path.exists():
                  with file_path.open("r", encoding="utf-8") as f:
                    line_count = sum(1 for _ in f)
                  print(f"{split}: exists | {line_count} lines")
        else:
            print(f"{split}: MISSING")
                

In [ ]:
# NOW DOING THE SAME THING BUT FOR IMAGE-TEXT FILES
if IMG_DIR.exists():
    files = list(IMG_DIR.iterdir())
    images = [p for p in files if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    texts = [p for p in files if p.suffix.lower() == ".txt"]
    print("Total files:", len(files))
    print("Image files:", len(images))
    print("Text files:", len(texts))
    print("Example filenames:", [p.name for p in sorted(files)[:10]])


In [ ]:
# NOW WE ARE GOING TO CHECK THE STRUCTURE OF OUR DATASET MORE SPECIFICALLY OUR FOLDER NAMED DATA
for folder_name in ["enem", "surface", "background", "thinking"]:
    file_path = DATA_DIR/folder_name/"train.txt"
    print(f"\n{'='*20}{folder_name}{'='*20}")
    if file_path.exists():
        with file_path.open("r",encoding="utf-8") as f:
            for i,line in enumerate(f):
                if line.strip():
                    print(line[:500])
                    break
    else:
        print("Train.txt not found")
    

In [ ]:

from pathlib import Path

ROOT = Path("/kaggle/input/datasets/aryansingh1964/project-3-nlp/dataset")
DATA_DIR = ROOT / "data"
IMG_DIR = ROOT / "img_text"

folders = ["enem", "surface", "background", "thinking"]
splits = ["train", "val", "test"]

# Read image and text filenames only once
files = [p for p in IMG_DIR.iterdir() if p.is_file()]

image_stems = {
    p.stem for p in files
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
}
text_stems = {
    p.stem for p in files
    if p.suffix.lower() == ".txt"
}

def read_ids(path):
    with path.open("r", encoding="utf-8") as f:
        return [
            line.split("\t", 1)[0].strip()
            for line in f if line.strip()
        ]

for split in splits:
    print(f"\n--- {split.upper()} ---")
    ids_by_folder = {}

    for folder in folders:
        path = DATA_DIR / folder / f"{split}.txt"

        if not path.exists():
            print(f"{folder}: FILE MISSING")
            continue

        ids = read_ids(path)
        ids_by_folder[folder] = ids

        print(
            f"{folder}: {len(ids)} records | "
            f"{len(ids) - len(set(ids))} duplicate IDs"
        )

    if len(ids_by_folder) == len(folders):
        reference = set(ids_by_folder["enem"])

        for folder, ids in ids_by_folder.items():
            print(f"IDs match enem in {folder}:",
                  set(ids) == reference)

        missing_images = reference - image_stems
        missing_texts = reference - text_stems

        print("Missing images:", len(missing_images))
        print("Missing text files:", len(missing_texts))
        print("Example missing image IDs:", sorted(missing_images)[:5])
        print("Example missing text IDs:", sorted(missing_texts)[:5])

print("\nTotal image files:", len(image_stems))
print("Total text files:", len(text_stems))


In [ ]:
ROOT = Path("/kaggle/input/datasets/aryansingh1964/project-3-nlp/dataset")
DATA_DIR = ROOT / "data"
IMG_DIR = ROOT / "img_text"

example_id = "2499"

# Read the original social-media post
text_path = IMG_DIR / f"{example_id}.txt"

print("ORIGINAL TEXT")
print("-" * 50)
print(text_path.read_text(encoding="utf-8"))

# Read the entity and sentiment annotations
annotation_path = DATA_DIR / "enem" / "train.txt"

with annotation_path.open("r", encoding="utf-8") as f:
    for line in f:
        if line.startswith(example_id + "\t"):
            entity_data = ast.literal_eval(
                line.split("\t", 1)[1].strip()
            )

            print("\nENTITIES AND SENTIMENT LABELS")
            print("-" * 50)

            for entity, sentiments in entity_data:
                print(f"Entity: {entity}")
                print(f"Sentiments: {sentiments}\n")

            break

In [ ]:
for folder_name in ["enem", "surface", "background", "thinking"]:
    file_path = DATA_DIR/folder_name/"train.txt"
    print(f"\n{'='*20}{folder_name}{'='*20}")
    if file_path.exists():
        with file_path.open("r",encoding="utf-8") as f:
            for i,line in enumerate(f):
                if line.strip():
                    print(line[:1200])
                    break

In [ ]:
ROOT = Path("/kaggle/input/datasets/aryansingh1964/project-3-nlp/dataset")
DATA_DIR = ROOT / "data"
IMG_DIR = ROOT / "img_text"

# Read the entity and sentiment annotations
records = []

with open(DATA_DIR / "enem" / "train.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        example_id, annotation = line.split("\t", 1)
        entities = ast.literal_eval(annotation)

        text_path = IMG_DIR / f"{example_id}.txt"
        image_path = IMG_DIR / f"{example_id}.jpg"

        # Skip entries whose paired files are unavailable
        if not text_path.exists() or not image_path.exists():
            continue

        text = text_path.read_text(encoding="utf-8").strip()

        for entity, sentiments in entities:
            records.append({
                "example_id": example_id,
                "text": text,
                "image_path": str(image_path),
                "entity": entity,
                "original_sentiment_labels": ", ".join(sentiments)
            })

# Reproducible random sample of 100 aspect instances
rng = random.Random(42)
pilot = pd.DataFrame(records)

if len(pilot) < 100:
    raise ValueError(f"Only {len(pilot)} eligible instances found.")

pilot = pilot.iloc[
    sorted(rng.sample(range(len(pilot)), 100))
].copy().reset_index(drop=True)

# Empty fields for manual annotation
pilot["text_evidence"] = ""
pilot["image_evidence"] = ""
pilot["aspect_explicitness"] = ""
pilot["sentiment_explicitness"] = ""
pilot["evidence_source"] = ""
pilot["confidence"] = ""
pilot["annotator_notes"] = ""

# Save the annotation template
output_path = "/kaggle/working/manual_pilot_100.csv"
pilot.to_csv(output_path, index=False)

print("Pilot instances:", len(pilot))
print("Unique posts:", pilot["example_id"].nunique())
print("Saved to:", output_path)

display(pilot[
    ["example_id", "text", "entity",
     "original_sentiment_labels",
     "aspect_explicitness", "sentiment_explicitness"]
].head(10))


In [ ]:

# Displaying the first 10 instances for manual review and modification

for i, row in pilot.head(10).iterrows():
    print(f"\n{'=' * 60}")
    print(f"Instance: {i + 1} | Post ID: {row['example_id']}")
    print(f"Text: {row['text']}")
    print(f"Aspect: {row['entity']}")
    print(f"Original sentiment labels: {row['original_sentiment_labels']}")
    print(f"Image path: {row['image_path']}")


In [ ]:
batch = pilot.head(10)
cols=2
rows=math.ceil(len(batch)/cols)
fig, axes = plt.subplots(rows, cols, figsize=(12, rows * 4))
axes = axes.flatten()
for ax,(_,rows) in zip(axes,batch.iterrows()):
    try:
        img = Image.open(row["image_path"]).convert("RGB")
        img.thumbnail((450,300))
        ax.imshow(img)
        ax.set_title(f"ID: {row['example_id']}\nAspect: {row['entity']}",fontsize=9)
    except  exception as e:
          ax.text(0.5, 0.5, f"Image unavailable\n{e}",ha="center", va="center")
    ax.axis("off")
for ax in axes[len(batch):]:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
IMG_DIR = ROOT / "img_text"

# Check the three unique IDs from your image description
for example_id in ["4833", "4835", "4845"]:
    text_path = IMG_DIR / f"{example_id}.txt"

    print(f"\n{'=' * 50}")
    print(f"Post ID: {example_id}")

    if text_path.exists():
        print(text_path.read_text(encoding="utf-8").strip())
    else:
        print("Text file not found.")


In [ ]:
for example_id in ["4833", "4835", "4845"]:
    text_path = IMG_DIR/f"{example_id}.txt"
    print("\n" + "=" *60)
    print(f"Post_ID {example_id}")
    if text_path.exists():
        text = text_path.read_text(encoding="utf-8").strip()
        print(text)
    else:
        print("Text not found")
        
    

In [ ]:
selected_ids = {"4833", "4835", "4845"}
enem_file = DATA_DIR / "enem" / "train.txt"

with open(enem_file, "r", encoding="utf-8") as file:
    for line in file:
        example_id, annotation = line.rstrip("\n").split("\t", 1)

        if example_id in selected_ids:
            entities = ast.literal_eval(annotation)

            print(f"\nPost ID: {example_id}")
            for entity, sentiments in entities:
                print(f"Aspect: {entity}")

In [ ]:
selected_ids = {"4833", "4835", "4845"}
enem_file = DATA_DIR / "enem" / "train.txt"

with open(enem_file, "r", encoding="utf-8") as file:
    for line in file:
        example_id, annotation = line.rstrip("\n").split("\t", 1)

        if example_id in selected_ids:
            entities = ast.literal_eval(annotation)

            print(f"\nPost ID: {example_id}")
            for entity, sentiments in entities:
                print(f"Aspect: {entity}")
                print(f"Sentiment labels: {sentiments}")

In [ ]:
selected_ids = ["4833", "4835", "4845"]

fig, axes = plt.subplots(1, 3, figsize=(15, 6))

for ax, example_id in zip(axes, selected_ids):
    image_path = IMG_DIR / f"{example_id}.jpg"

    if image_path.exists():
        with Image.open(image_path) as img:
            ax.imshow(img.convert("RGB"))
        ax.set_title(f"Post ID: {example_id}")
    else:
        ax.set_title(f"Image not found: {example_id}")

    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# SETTING UP GPT-OSS TO ANNOTATE FIRST 5 POSTS 
for root, dirs, files in os.walk("/kaggle/input"): # verifing to check weather the files are uploaded or not
    for file in files:
        if file in ["pilot_100.jsonl","annotation_prompt.md","annotation_guidelines.md"]:
            print(os.path.join(root, file))

In [ ]:
pilot_files = glob.glob("/kaggle/input/**/pilot_100.jsonl",recursive=True)
assert len(pilot_files) == 1, (f"Expected exactly one pilot file, found: {pilot_files}")
pilot_path = pilot_files[0]
with open(pilot_path, "r", encoding="utf-8") as f:
    pilot_data = [json.loads(line) for line in f if line.strip()]
print("Total pilot posts:", len(pilot_data))
print("\nFirst record:")
print(json.dumps(pilot_data[0], ensure_ascii=False, indent=2))

In [ ]:
# create an output dictnary
output_dir = Path("/kaggle/working/gpt_oss_pilot_test")
output_dir.mkdir(parents=True, exist_ok=True)
print("Output directory created:", output_dir)

In [ ]:
# SELECT THE FIRST 5 POSTS
test_posts = pilot_data[:5]
print("Number of posts selected:", len(test_posts))
print("Selected post IDs:", [post["pilot_id"] for post in test_posts])

In [ ]:
# PREPARE THE MODEL FOR ANNOTATION
system_instruction = (
    annotation_prompt
    + "\n\n"
    + annotation_guidelines
)
model.eval()
raw_responses = []

In [ ]:
for post_number, post in enumerate(test_posts, start=1):
    print(f"\nAnnotating {post['pilot_id']} "
          f"({post['num_aspects']} aspects)...")
    candidate_aspects = [
        {
            "aspect_index": index,
            "aspect": aspect["aspect"]
        }
        for index, aspect in enumerate(post["aspects"])
    ]
    user_input = {
        "pilot_id": post["pilot_id"],
        "sample_id": post["sample_id"],
        "post_text": post["text"],
        "candidate_aspects": candidate_aspects
    }
    messages = [
        {
            "role": "system",
            "content": system_instruction
        },
        {
            "role": "user",
            "content": json.dumps(
                user_input,
                ensure_ascii=False,
                indent=2
            )
        }
    ]
    inputs = tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,return_tensors="pt",return_dict=True)
    inputs = {key: value.to("cuda:0")for key, value in inputs.items()}
    input_length = inputs["input_ids"].shape[1]
    with torch.inference_mode():
        output = model.generate(**inputs,max_new_tokens=2500,do_sample=False)
    generated_tokens = output[0][input_length:]
    raw_output = tokenizer.decode(generated_tokens,skip_special_tokens=False)
    raw_responses.append({"pilot_id": post["pilot_id"],"sample_id": post["sample_id"],"raw_response": raw_output})
    output_file = output_dir / f"{post['pilot_id']}_raw.txt"
    with open(output_file, "w", encoding="utf-8") as file:
        file.write(raw_output)
    print("Raw response saved:", output_file)
    print("Generation completed.")
    del inputs, output, generated_tokens
    torch.cuda.empty_cache()

In [ ]:
combined_file = output_dir / "five_post_raw_responses.json"

with open(combined_file, "w", encoding="utf-8") as file:
    json.dump(raw_responses,file,ensure_ascii=False,indent=2)
print("Output directory:", output_dir)
print("Combined responses saved:", combined_file)